# v0.3 opponent completion run (Kaggle, GPU T4 x2, 2026-10-31)
Google TimesFM 2.5 / 3.0 zero-shot on PEMS04 and IMD central-India rainfall, and TimesFM 3.0 univariate on PEMS08 - the rounds the Colab session could not finish.

In [ ]:
import os, torch; print("CUDA:", torch.cuda.is_available(), torch.__version__)
assert torch.cuda.is_available(), "no GPU - set Accelerator to GPU"
for d in ["pwm","opponents","data/raw","results"]: os.makedirs(f"/kaggle/working/pwm_kit/{d}", exist_ok=True)
open("/kaggle/working/pwm_kit/pwm/__init__.py","w").write("")
open("/kaggle/working/pwm_kit/pwm/data.py","w").write(r'''"""PWM data layer: real physical-world observations on a spatial graph.

Every dataset becomes the same structure:
  raw     [T, N]  observations in real units (NaN = missing / not observed)
  A       [N, N]  row-normalised physical adjacency (road distances or geographic grid)
  period  index of the observation inside its natural cycle (5-min slot of day / day of year)
  dow     day of week (0 for daily climate data, where it carries no meaning)

Datasets
  PEMS08, PEMS04   Caltrans road-sensor traffic flow, 5-min steps (standard benchmark protocol:
                   12 -> 12 steps, chronological 60/20/20 split, zero-masked metrics)
  IMD_<region>     India Meteorological Department 0.25 deg gridded daily rainfall (mm/day),
                   30 days -> next 7 days, split by year, zeros are REAL (no zero-masking)
"""
import datetime as dt
import os

import numpy as np
import torch

ROOT = os.path.dirname(os.path.dirname(os.path.abspath(__file__)))
RAW = os.path.join(ROOT, "data", "raw")

# lat_min, lat_max, lon_min, lon_max  (0.25 deg IMD grid)
IMD_REGIONS = {
    "central_india": (21.0, 26.75, 74.0, 82.75),   # Madhya Pradesh and surroundings
    "north_india":   (26.0, 31.75, 74.0, 80.75),   # Delhi NCR, Haryana, west UP, Punjab
    "west_coast":    (14.0, 20.75, 72.5, 76.25),   # Mumbai, Konkan, Western Ghats
}


def _row_norm(A):
    np.fill_diagonal(A, 1.0)
    return torch.tensor(A / A.sum(1, keepdims=True), dtype=torch.float32)


class SeriesData:
    def __init__(self, name, raw, A, period_idx, dow, period, in_len, out_len, bounds,
                 transform="none", mask_zero=False, units="", meta=None):
        self.name, self.raw, self.A = name, raw.astype(np.float32), A
        self.period_idx, self.dow, self.period = period_idx.astype(np.int64), dow.astype(np.int64), period
        self.in_len, self.out_len, self.bounds = in_len, out_len, bounds
        self.transform, self.mask_zero, self.units, self.meta = transform, mask_zero, units, meta or {}
        self.T, self.N = raw.shape
        self.observed = ~np.isnan(self.raw)
        f = self.fwd(np.nan_to_num(self.raw, nan=0.0))
        a, b = bounds["train"]
        tr = f[:b][self.observed[:b]]
        self.mean, self.std = float(tr.mean()), float(tr.std())   # TRAIN split only
        self.norm = np.where(self.observed, (f - self.mean) / self.std, 0.0).astype(np.float32)

    def fwd(self, x):
        return np.log1p(np.maximum(x, 0)) if self.transform == "log1p" else x

    def inv(self, z):   # z: normalised model output (torch) -> real units
        v = z * self.std + self.mean
        return torch.clamp(torch.expm1(v), min=0) if self.transform == "log1p" else v

    def indices(self, split, stride=1):
        a, b = self.bounds[split]
        return np.arange(a + self.in_len, b - self.out_len + 1)[::stride]

    def valid(self, y):
        """Which ground-truth values count in metrics."""
        m = ~np.isnan(y)
        if self.mask_zero:
            m &= y > 1e-4
        return m

    def batch(self, idx, device="cpu"):
        hi = idx[:, None] + np.arange(-self.in_len, 0)[None]
        fo = idx[:, None] + np.arange(self.out_len)[None]
        x = torch.from_numpy(self.norm[hi]).to(device)
        obs = torch.from_numpy(self.observed[hi]).to(device)
        y = torch.from_numpy(self.raw[fo]).to(device)                 # real units, NaN = unknown
        p = torch.from_numpy(self.period_idx[idx - 1]).to(device)      # cycle position of last obs
        d = torch.from_numpy(self.dow[idx - 1]).to(device)
        return x, obs, y, p, d

    def describe(self):
        n = [len(self.indices(s)) for s in ("train", "val", "test")]
        return (f"{self.name}: {self.T} steps x {self.N} locations, "
                f"{int((self.A > 0).sum()) - self.N} links; train/val/test windows = {n}")


# --------------------------------------------------------------------------- PeMS traffic
PEMS_START = {"PEMS08": dt.date(2016, 7, 1), "PEMS04": dt.date(2018, 1, 1)}


def load_pems(name):
    raw = np.load(f"{RAW}/{name}.npz")["data"][..., 0].astype(np.float32)
    T, N = raw.shape
    edges = np.atleast_2d(np.genfromtxt(f"{RAW}/{name}.csv", delimiter=",", skip_header=1))
    dist = np.full((N, N), np.inf)
    for s, t, c in edges:
        dist[int(s), int(t)] = dist[int(t), int(s)] = c
    sigma = dist[np.isfinite(dist)].std()
    A = np.exp(-(dist / sigma) ** 2)
    A[A < 0.1] = 0.0
    t = np.arange(T)
    dow = (t // 288 + PEMS_START[name].weekday()) % 7
    n_tr, n_va = int(T * 0.6), int(T * 0.2)
    bounds = {"train": (0, n_tr), "val": (n_tr, n_tr + n_va), "test": (n_tr + n_va, T)}
    return SeriesData(name, raw, _row_norm(A), t % 288, dow, 288, 12, 12, bounds,
                      transform="none", mask_zero=True, units="vehicles / 5 min")


# --------------------------------------------------------------------------- IMD rainfall
def imd_cache_path(region, y0, y1):
    return f"{RAW}/IMD_rain_{region}_{y0}_{y1}.npz"


def download_imd(region="central_india", y0=2000, y1=2024):
    """Download IMD 0.25 deg daily gridded rainfall with imdlib and cache the region as .npz."""
    import imdlib as imd
    la0, la1, lo0, lo1 = IMD_REGIONS[region]
    obj = imd.load("rain", y0, y1, cache_dir=os.path.join(RAW, "imd_cache"))
    ds = obj.get_xarray()
    var = list(ds.data_vars)[0]
    da = ds[var]
    dims = {d.lower(): d for d in da.dims}
    tdim = next(dims[k] for k in dims if "time" in k or "date" in k)
    ladim = next(dims[k] for k in dims if "lat" in k)
    lodim = next(dims[k] for k in dims if "lon" in k)
    da = da.sortby(ladim).sortby(lodim)
    da = da.sel({ladim: slice(la0, la1), lodim: slice(lo0, lo1)}).transpose(tdim, ladim, lodim)
    if da.sizes[ladim] == 0 or da.sizes[lodim] == 0:
        raise RuntimeError(f"region {region} selected no IMD grid cells")
    arr = da.values.astype(np.float32)
    arr[arr < 0] = np.nan                                    # IMD fill value (-999) -> missing
    lat, lon = da[ladim].values, da[lodim].values
    dates = np.array([str(x)[:10] for x in da[tdim].values])
    np.savez_compressed(imd_cache_path(region, y0, y1), rain=arr, lat=lat, lon=lon, dates=dates)
    return imd_cache_path(region, y0, y1)


def load_imd(region="central_india", y0=2000, y1=2024, in_len=30, out_len=7, path=None, name=None):
    path = path or imd_cache_path(region, y0, y1)
    if not os.path.exists(path):
        raise FileNotFoundError(f"{path} missing - run download_imd('{region}', {y0}, {y1}) first")
    z = np.load(path)
    arr, lat, lon, dates = z["rain"], z["lat"], z["lon"], z["dates"]
    Tn, H, W = arr.shape
    flat = arr.reshape(Tn, H * W)
    keep = np.where(np.mean(~np.isnan(flat), 0) > 0.9)[0]  # land cells with >90% coverage
    raw = flat[:, keep]
    ii, jj = np.unravel_index(keep, (H, W))
    clat, clon = lat[ii], lon[jj]
    # geographic graph: 8-neighbour cells on the 0.25 deg grid, weight decays with distance
    dlat = (clat[:, None] - clat[None]) * 111.0
    dlon = (clon[:, None] - clon[None]) * 111.0 * np.cos(np.deg2rad(clat[:, None]))
    km = np.sqrt(dlat ** 2 + dlon ** 2)
    A = np.where(km < 0.25 * 111 * 1.5, np.exp(-(km / 30.0) ** 2), 0.0)
    d = [dt.date.fromisoformat(s) for s in dates]
    doy = np.array([x.timetuple().tm_yday - 1 for x in d])
    years = np.array([x.year for x in d])
    yl = years.max()
    te0 = int(np.argmax(years >= yl - 2))
    va0 = int(np.argmax(years >= yl - 5))
    bounds = {"train": (0, va0), "val": (va0, te0), "test": (te0, Tn)}
    return SeriesData(name or f"IMD_{region}", raw, _row_norm(A), doy, np.zeros(Tn), 366, in_len, out_len,
                      bounds, transform="log1p", mask_zero=False, units="mm / day",
                      meta={"lat": clat.tolist(), "lon": clon.tolist(),
                            "years": [int(years.min()), int(yl)],
                            "test_years": [int(yl - 2), int(yl)], "val_years": [int(yl - 5), int(yl - 3)]})


def load(name, **kw):
    if name.startswith("PEMS"):
        return load_pems(name)
    if name == "SYNTHETIC_IMD":   # test-only data, never reported
        return load_imd(path=os.path.join(RAW, "SYNTHETIC_IMD_rain_test.npz"), name="SYNTHETIC_IMD")
    if name.startswith("IMD_"):
        return load_imd(name[4:], **kw)
    raise ValueError(name)
''')

open("/kaggle/working/pwm_kit/pwm/evaluation.py","w").write(r'''"""Shared scoring: every model (PWM or an opponent) is scored by this one function on
exactly the same windows, so comparisons are like-for-like."""
import numpy as np


def score(q10, q50, q90, y, valid):
    """All arrays [S, H, N] in real units. valid = which truths count."""
    e = (q50 - y)[valid]
    yt = y[valid]
    nz = yt > 1e-4
    out = {
        "MAE": float(np.abs(e).mean()),
        "RMSE": float(np.sqrt((e ** 2).mean())),
        "MAPE": float((np.abs(e[nz]) / yt[nz]).mean() * 100),
        "per_horizon_MAE": [float(np.abs((q50[:, h] - y[:, h])[valid[:, h]]).mean())
                            for h in range(y.shape[1])],
        "n_values_scored": int(valid.sum()),
    }
    if q10 is not None:
        inside = (y >= q10) & (y <= q90)
        out["interval_80_coverage"] = float(inside[valid].mean())
        out["interval_80_mean_width"] = float((q90 - q10)[valid].mean())
    return out


def simple_baselines(ds, idx):
    """Last value and seasonal historical average (from TRAIN data only)."""
    fo = idx[:, None] + np.arange(ds.out_len)[None]
    y = ds.raw[fo]
    valid = ds.valid(y)
    last = np.repeat(np.nan_to_num(ds.raw[idx - 1])[:, None], ds.out_len, 1)
    _, b = ds.bounds["train"]
    key = ds.dow[:b] * ds.period + ds.period_idx[:b]
    table = np.full((7 * ds.period, ds.N), np.nan, np.float32)
    for k in np.unique(key):
        table[k] = np.nanmean(ds.raw[:b][key == k], 0)
    table = np.where(np.isnan(table), np.nanmean(ds.raw[:b], 0), table)
    ha = table[ds.dow[fo] * ds.period + ds.period_idx[fo]]
    return {"last_value": score(None, last, None, y, valid),
            "historical_average": score(None, ha, None, y, valid)}
''')

open("/kaggle/working/pwm_kit/opponents/run_timesfm.py","w").write(r'''"""OPPONENT benchmark: Google TimesFM, zero-shot, on exactly the same test windows as PWM.

Run ONLY with the separate opponents environment's python. This file imports the shared
data loader and scoring function from pwm/, but never PWM's model, and nothing here is ever
saved into or mixed with PWM weights.

  opponents_env/bin/python opponents/run_timesfm.py --data PEMS08 --model tfm25 --ctx 512
  opponents_env/bin/python opponents/run_timesfm.py --data PEMS08 --model tfm3  --ctx 512 --mode multivariate

Fairness choices (all in TimesFM's favour or neutral):
  * TimesFM gets a LONG history (default 512 steps) while PWM sees only 12 (PeMS) / 30 (IMD).
  * TimesFM 3.0 can also run multivariate, letting it look at other locations too.
  * Scoring uses the same function, windows and masking as PWM.
Licences: TimesFM 2.5 weights = Apache-2.0. TimesFM 3.0 weights = non-commercial research only.
"""
import argparse, json, os, sys, time
import numpy as np

HERE = os.path.dirname(os.path.abspath(__file__))
sys.path.insert(0, os.path.dirname(HERE))
from pwm.data import load, ROOT          # data + scoring only
from pwm.evaluation import score


def contexts_for(ds, idx, ctx):
    """[W, N, ctx] real-unit histories ending just before each window (NaN gaps forward-filled)."""
    lo = idx[:, None] + np.arange(-ctx, 0)[None]
    lo = np.clip(lo, 0, None)
    c = ds.raw[lo]                                            # [W, ctx, N]
    c = c.transpose(0, 2, 1).copy()
    for w in range(c.shape[0]):                               # forward fill, then 0
        x = c[w]
        mask = np.isnan(x)
        if mask.any():
            idxs = np.where(~mask, np.arange(x.shape[1]), 0)
            np.maximum.accumulate(idxs, axis=1, out=idxs)
            c[w] = np.nan_to_num(x[np.arange(x.shape[0])[:, None], idxs], nan=0.0)
    return c.astype(np.float32)


class TFM25:
    name, licence = "TimesFM-2.5 (Google, 200M)", "Apache-2.0"

    def __init__(self, ctx, H, bs):
        import timesfm, torch
        torch.set_float32_matmul_precision("high")
        self.m = timesfm.TimesFM_2p5_200M_torch.from_pretrained("google/timesfm-2.5-200m-pytorch")
        self.m.compile(timesfm.ForecastConfig(
            max_context=int(np.ceil(ctx / 32) * 32), max_horizon=128, normalize_inputs=True,
            use_continuous_quantile_head=True, force_flip_invariance=True,
            infer_is_positive=True, fix_quantile_crossing=True, per_core_batch_size=bs))
        self.H = H

    def __call__(self, series):                              # list of 1-D arrays
        point, q = self.m.forecast(horizon=self.H, inputs=series)
        return q[..., 1], q[..., 5], q[..., 9]                # 10th, median, 90th percentile


class TFM3:
    name, licence = "TimesFM-3.0 (Google)", "non-commercial research licence"

    def __init__(self, ctx, H, bs, mode):
        import torch
        from timesfm3 import TimesFM3Evaluator, ModelConfig
        dev = "cuda" if torch.cuda.is_available() else "cpu"
        self.m = TimesFM3Evaluator(ModelConfig(checkpoint_path="google/timesfm-3.0-pytorch",
                                               per_core_batch_size=bs, device=dev))
        self.H, self.mode = H, mode
        if mode == "multivariate":
            self.name += " multivariate"

    def __call__(self, series):
        if self.mode == "multivariate":                       # series: list of [N, ctx]
            outs = list(self.m.predict_batch(contexts=series, horizon=self.H, return_quantiles=True))
            q = np.stack([o.quantiles for o in outs])          # [W, N, H, 9]
            q = q.reshape(-1, self.H, q.shape[-1])
        else:
            outs = list(self.m.predict_batch(contexts=series, horizon=self.H,
                                             return_quantiles=True, univariate=True))
            q = np.stack([o.quantiles for o in outs])          # [B, H, 9]
        return q[..., 0], q[..., 4], q[..., 8]


class Stub:
    """TEST ONLY - lets the pipeline be checked without downloading weights. Never reported."""
    name, licence = "STUB (pipeline test, not a model)", "n/a"

    def __init__(self, ctx, H, bs, mode="univariate"):
        self.H, self.mode = H, mode

    def __call__(self, series):
        s = np.stack(series)
        if self.mode == "multivariate":
            s = s.reshape(-1, s.shape[-1])
        last = np.repeat(s[:, -1:], self.H, 1)
        return last * 0.8, last, last * 1.2


def main(a):
    ds = load(a.data)
    idx = ds.indices("test", a.stride)
    if a.model == "stub":
        fc = Stub(a.ctx, ds.out_len, a.bs, a.mode)
    elif a.model == "tfm25":
        fc = TFM25(a.ctx, ds.out_len, a.bs)
    else:
        fc = TFM3(a.ctx, ds.out_len, a.bs, a.mode)
    print(f"{fc.name} on {ds.name}: {len(idx)} windows x {ds.N} locations, context {a.ctx}", flush=True)
    W, H, N = len(idx), ds.out_len, ds.N
    q10, q50, q90 = (np.zeros((W, H, N), np.float32) for _ in range(3))
    t0 = time.time()
    for s in range(0, W, a.chunk):
        c = contexts_for(ds, idx[s:s + a.chunk], a.ctx)       # [w, N, ctx]
        w = len(c)
        series = list(c) if a.mode == "multivariate" and a.model in ("tfm3", "stub") \
            else list(c.reshape(w * N, a.ctx))
        lo, mid, hi = fc(series)
        for arr, v in ((q10, lo), (q50, mid), (q90, hi)):
            arr[s:s + w] = np.asarray(v, np.float32).reshape(w, N, H).transpose(0, 2, 1)
        if (s // a.chunk) % 10 == 0:
            print(f"  {min(s + a.chunk, W)}/{W} windows ({(time.time()-t0)/60:.1f} min)", flush=True)
    y = ds.raw[idx[:, None] + np.arange(H)[None]]
    res = score(q10, q50, q90, y, ds.valid(y))
    tag = {"tfm25": "TimesFM-2.5", "tfm3": "TimesFM-3.0", "stub": "STUB"}[a.model]
    if a.model == "tfm3" and a.mode == "multivariate":
        tag += "-multivariate"
    out = {"model": fc.name, "provenance": f"external opponent, zero-shot, weights licence: {fc.licence}",
           "is_stub": a.model == "stub", "dataset": ds.name, "context_steps": a.ctx,
           "stride": a.stride, "test": res, "minutes": (time.time() - t0) / 60}
    os.makedirs(os.path.join(ROOT, "results"), exist_ok=True)
    json.dump(out, open(os.path.join(ROOT, "results", f"{ds.name}__{tag}.json"), "w"), indent=1)
    print(f"{fc.name}: test MAE {res['MAE']:.3f} RMSE {res['RMSE']:.3f} "
          f"cov80 {res['interval_80_coverage']*100:.1f}%")


if __name__ == "__main__":
    ap = argparse.ArgumentParser()
    ap.add_argument("--data", required=True)
    ap.add_argument("--model", default="tfm25", choices=["tfm25", "tfm3", "stub"])
    ap.add_argument("--mode", default="univariate", choices=["univariate", "multivariate"])
    ap.add_argument("--ctx", type=int, default=512)
    ap.add_argument("--stride", type=int, default=1)
    ap.add_argument("--chunk", type=int, default=32, help="windows per forward batch")
    ap.add_argument("--bs", type=int, default=256)
    main(ap.parse_args())
''')

print("files written")
%cd /kaggle/working/pwm_kit
!pip install -q imdlib timesfm 2>&1 | tail -1
!python -c "import timesfm, timesfm3, torch; print('timesfm OK cuda', torch.cuda.is_available())"
!for d in PEMS08 PEMS04; do for e in npz csv; do curl -sL -o data/raw/$d.$e https://raw.githubusercontent.com/guoshnBJTU/ASTGNN/main/data/$d/$d.$e; done; done; ls -la data/raw
from pwm.data import download_imd; print("IMD:", download_imd("central_india", 2000, 2024))
OPP = "python opponents/run_timesfm.py"
!{OPP} --data PEMS04 --model tfm25 --ctx 512 --stride 2 2>&1 | tail -1
!{OPP} --data PEMS04 --model tfm3 --mode multivariate --ctx 512 --stride 2 2>&1 | tail -1
!{OPP} --data IMD_central_india --model tfm25 --ctx 365 --stride 2 2>&1 | tail -1
!{OPP} --data IMD_central_india --model tfm3 --mode multivariate --ctx 365 --stride 2 2>&1 | tail -1
!{OPP} --data PEMS08 --model tfm3 --mode univariate --ctx 512 --stride 1 2>&1 | tail -1
import glob, json
print("=== OPPONENT RESULTS ===")
for f in sorted(glob.glob("results/*.json")):
    d=json.load(open(f)); t=d["test"]; print(os.path.basename(f), "->", d["model"], "MAE", round(t["MAE"],3), "RMSE", round(t["RMSE"],3), "MAPE", round(t["MAPE"],2), "cov80", round(t["interval_80_coverage"]*100,1))
print("KAGGLE-DONE")